# LeetCode 710 — Random Pick with Blacklist bằng Python

- Link: https://leetcode.com/problems/random-pick-with-blacklist/
- Độ khó: **Hard**
- Chủ đề: Ngẫu nhiên đều (uniform random), Bảng băm (remapping), Tìm kiếm nhị phân (`bisect`), Thiết kế lớp

Skeleton trên LeetCode (Python3):

```python
class Solution:

    def __init__(self, n: int, blacklist: List[int]):

    def pick(self) -> int:


# Your Solution object will be instantiated and called as such:
# obj = Solution(n, blacklist)
# param_1 = obj.pick()
```

## 1. Giới thiệu & mục tiêu học

Ta cần một "máy rút thăm" trả về số nguyên ngẫu nhiên **đều** trong `[0, n)` nhưng **tránh** các số trong danh sách đen. Khó ở chỗ `n` có thể tới `10^9` (không thể liệt kê hết số hợp lệ) và danh sách đen có thể rất dài (rút rồi loại bỏ có thể phải rút lại hàng trăm nghìn lần).

Mẹo đẹp nhất của bài: **chỉ rút trong `[0, m)`** với `m` = số lượng số hợp lệ, rồi **"đổi tên"** những số đen nằm trong `[0, m)` thành những số trắng nằm ngoài `[0, m)`. Mỗi lần `pick` chỉ còn `O(1)`.

Sau notebook này bạn sẽ:

1. Hiểu "ngẫu nhiên đều" nghĩa là gì và vì sao một **song ánh** từ `[0, m)` sang tập số trắng cho ra phân phối đều.
2. Viết hai cách ngây thơ (liệt kê danh sách trắng; rút-loại) và thấy giới hạn của chúng.
3. Cài đặt cách **ánh xạ lại** (remapping) bằng dict: khởi tạo `O(B)`, `pick` `O(1)`.
4. Cài đặt cách **tìm kiếm nhị phân**: tìm số trắng thứ `k` trong `O(log B)`.
5. Kiểm thử chương trình ngẫu nhiên đúng cách: **seed cố định**, kiểm tra song ánh, và kiểm tra phân phối thô.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho số nguyên `n` và mảng `blacklist` gồm các số nguyên **phân biệt** trong `[0, n)`. Thiết kế lớp:

- `Solution(n, blacklist)`: khởi tạo.
- `pick()`: trả về một số nguyên ngẫu nhiên trong `[0, n)` **không** nằm trong `blacklist`. Mọi số hợp lệ phải có **xác suất bằng nhau**.

Yêu cầu tối ưu: **giảm số lần gọi hàm sinh số ngẫu nhiên có sẵn** của ngôn ngữ (lý tưởng: đúng 1 lần mỗi `pick`).

**Ví dụ**

```text
Input:
["Solution", "pick", "pick", "pick", "pick", "pick", "pick", "pick"]
[[7, [2, 3, 5]], [], [], [], [], [], [], []]
Output:
[null, 0, 4, 1, 6, 1, 0, 4]
Giải thích: số hợp lệ là 0, 1, 4, 6; mỗi số được trả về với xác suất 1/4.
            (Output cụ thể là ngẫu nhiên, chỉ là một khả năng.)
```

**Ràng buộc**

- `1 <= n <= 10^9`
- `0 <= blacklist.length <= min(10^5, n - 1)`
- `0 <= blacklist[i] < n`, các phần tử đôi một khác nhau
- Tối đa `2 * 10^4` lần gọi `pick`.

In [1]:
import random

n, blacklist = 7, [2, 3, 5]
white = [x for x in range(n) if x not in set(blacklist)]
print("số hợp lệ:", white, "| xác suất mỗi số mong muốn:", 1 / len(white))

số hợp lệ: [0, 1, 4, 6] | xác suất mỗi số mong muốn: 0.25


## 3. Kiến thức nền

### 3.1. `random.randrange` và `random.seed`

- `random.randrange(m)` trả về số nguyên đều trong `[0, m)` (không gồm `m`).
- `random.randint(a, b)` trả về số trong `[a, b]` — **gồm cả `b`** (dễ nhầm!).
- `random.seed(s)` đặt "hạt giống": cùng seed → cùng dãy số. Nhờ vậy test chương trình ngẫu nhiên vẫn **lặp lại được**.

In [2]:
random.seed(42)
a = [random.randrange(4) for _ in range(10)]
random.seed(42)
b = [random.randrange(4) for _ in range(10)]
print(a, a == b)
print("randint(0, 3) có thể ra 3:", 3 in [random.randint(0, 3) for _ in range(100)])

[0, 0, 2, 1, 1, 1, 0, 0, 3, 0] True
randint(0, 3) có thể ra 3: True


### 3.2. "Đều" và song ánh

Nếu `k` đều trên `[0, m)` (mỗi giá trị xác suất `1/m`) và `f` là **song ánh** (mỗi đầu vào ra một đầu ra khác nhau, phủ hết tập đích `T` có `m` phần tử), thì `f(k)` đều trên `T`: mỗi `t ∈ T` có đúng một `k` với `f(k) = t`, nên `P(f(k) = t) = 1/m`.

Vậy bài toán quy về: **xây một song ánh rẻ từ `[0, m)` sang tập số trắng**.

In [3]:
m = 4
f = {0: 0, 1: 1, 2: 6, 3: 4}            # một song ánh [0,4) -> {0,1,4,6}
print(sorted(f.values()) == white)

True


### 3.3. `set` để kiểm tra thuộc tập trong `O(1)`

`x in some_set` trung bình `O(1)`, còn `x in some_list` là `O(len)`.

In [4]:
bl = set([2, 3, 5])
print(3 in bl, 4 in bl)

True False


### 3.4. `dict.get(key, default)`

Trả về `d[key]` nếu có, ngược lại trả `default`. Rất hợp cho "nếu số này được đổi tên thì lấy tên mới, không thì giữ nguyên".

In [5]:
remap = {2: 6, 3: 4}
print([remap.get(k, k) for k in range(4)])

[0, 1, 6, 4]


### 3.5. `bisect_right` trên mảng không giảm

`bisect_right(a, x)` = số phần tử `≤ x` (với `a` sắp không giảm).

In [6]:
from bisect import bisect_right
print(bisect_right([2, 2, 3], 1), bisect_right([2, 2, 3], 2), bisect_right([2, 2, 3], 9))

0 2 3


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Đếm trước

Có `B = len(blacklist)` số đen, nên có `m = n - B` số trắng. Mỗi `pick` cần rút **một** số `k` đều trong `[0, m)` rồi biến nó thành một số trắng.

### 4.2. Tính tay với `n = 7`, `blacklist = [2, 3, 5]`

`m = 4`. Chia trục thành hai vùng:

```text
vị trí :  0  1  2  3 | 4  5  6
màu    :  W  W  B  B | W  B  W
          vùng [0, m)  | vùng [m, n)
```

- Trong vùng `[0, 4)`: có 2 số đen (2, 3) — "lỗ".
- Trong vùng `[4, 7)`: có 2 số trắng (4, 6) — "dư".

Số lỗ **luôn bằng** số dư! (Vì vùng `[m, n)` có `B` ô, trong đó số đen = `B - (số đen trong [0,m))`, nên số trắng trong đó = số đen trong `[0, m)`.)

→ Lấp mỗi lỗ bằng một số dư: `2 → 4`, `3 → 6`. Rút `k ∈ [0, 4)`: `0→0, 1→1, 2→4, 3→6`. Đó là một song ánh sang `{0, 1, 4, 6}`.

### 4.3. Cách nhìn thứ hai: "số trắng thứ k"

Sắp xếp số trắng: `[0, 1, 4, 6]`. Rút `k` rồi trả **số trắng thứ `k`** (đếm từ 0). Không cần lưu danh sách trắng nếu tính được "số trắng thứ `k`" nhanh — dùng tìm kiếm nhị phân trên danh sách đen.

In [7]:
n, bl = 7, [2, 3, 5]
m = n - len(bl)
holes = [b for b in bl if b < m]
spares = [x for x in range(m, n) if x not in set(bl)]
print("m =", m, "| lỗ:", holes, "| dư:", spares)
print("ánh xạ:", dict(zip(holes, spares)))

m = 4 | lỗ: [2, 3] | dư: [4, 6]
ánh xạ: {2: 4, 3: 6}


## 5. Cách 1 — Brute Force: liệt kê danh sách trắng

**Ý tưởng.** Khởi tạo: tạo list mọi số trắng. `pick`: `random.choice` (hoặc `white[randrange(len(white))]`).

Đơn giản, đúng tuyệt đối, chỉ gọi random một lần — nhưng tốn `O(n)` thời gian và bộ nhớ khi khởi tạo.

In [8]:
class PickWhitelist:
    def __init__(self, n, blacklist):
        black = set(blacklist)
        self.white = [x for x in range(n) if x not in black]

    def pick(self):
        return self.white[random.randrange(len(self.white))]

random.seed(1)
p = PickWhitelist(7, [2, 3, 5])
print([p.pick() for _ in range(12)])

[1, 0, 4, 0, 6, 6, 6, 6, 1, 0, 6, 0]


**Độ phức tạp.** Khởi tạo `O(n)` thời gian, `O(n)` bộ nhớ; `pick` `O(1)`.

**Nút thắt.** `n = 10^9` → list một tỷ phần tử (hàng chục GB). Không khả thi.

### 5.2. Cách 1b — Rút rồi loại (rejection sampling)

**Ý tưởng.** Rút `x` đều trong `[0, n)`; nếu `x` đen thì rút lại. Đúng (phân phối đều trên số trắng), bộ nhớ `O(B)`.

In [9]:
class PickRejection:
    def __init__(self, n, blacklist):
        self.n = n
        self.black = set(blacklist)

    def pick(self):
        while True:
            x = random.randrange(self.n)
            if x not in self.black:
                return x

random.seed(1)
p = PickRejection(7, [2, 3, 5])
print([p.pick() for _ in range(12)])
print("số lần rút trung bình mỗi pick khi n=10^5, B=10^5-1:", 10**5 / 1)

[1, 4, 6, 6, 6, 0, 0, 6, 6, 1, 0, 0]
số lần rút trung bình mỗi pick khi n=10^5, B=10^5-1: 100000.0


**Nút thắt.** Số lần rút trung bình là `n / (n - B)`. Khi `B = n - 1` (ràng buộc cho phép), trung bình phải rút `n` lần cho mỗi `pick` — với `n = 10^5` và `2·10^4` lần `pick` là `2·10^9` lần rút. Và đề yêu cầu **ít lần gọi random**.

## 6. Bước đột phá

**Insight:** rút trong `[0, m)` thay vì `[0, n)` — **luôn đúng một lần gọi random**. Còn lại chỉ là xây song ánh `f: [0, m) → White`.

**Cách xây 1 (ánh xạ lại):**

> **Nếu** `k` trắng **thì** `f(k) = k`. **Nếu** `k` đen (một "lỗ" trong `[0, m)`) **thì** `f(k)` = một số trắng trong `[m, n)` chưa dùng.

Chỉ cần lưu dict cho các lỗ: ≤ `B` cặp. Bất biến khi xây: con trỏ `w` chạy tăng dần trong `[m, n)`, luôn bỏ qua số đen, mỗi số trắng ở đó được gán cho đúng một lỗ.

**Cách xây 2 (số trắng thứ k):** với danh sách đen đã sắp `b[0] < b[1] < …`, số trắng **nhỏ hơn** `b[j]` là `b[j] - j`. Gọi `key[j] = b[j] - j` (không giảm). Khi đó:

> số trắng thứ `k` = `k + c`, với `c` = số `j` có `key[j] ≤ k` = `bisect_right(key, k)`.

(Trực giác: đi từ `k`, mỗi số đen "chen" trước số trắng thứ `k` làm nó dịch phải 1.)

In [10]:
b = [2, 3, 5]
key = [b[j] - j for j in range(len(b))]
print("key =", key)
for k in range(4):
    c = bisect_right(key, k)
    print(f"k={k}: c={c} -> số trắng thứ k = {k + c}")

key = [2, 2, 3]
k=0: c=0 -> số trắng thứ k = 0
k=1: c=0 -> số trắng thứ k = 1
k=2: c=2 -> số trắng thứ k = 4
k=3: c=3 -> số trắng thứ k = 6


## 7. Các cách tiếp theo

Tổng cộng 4 cách:

| Cách | Khởi tạo | `pick` | Bộ nhớ | Lần gọi random/`pick` |
|---|---|---|---|---|
| 1. Danh sách trắng | `O(n)` | `O(1)` | `O(n)` | 1 |
| 1b. Rút-loại | `O(B)` | kỳ vọng `O(n/(n-B))` | `O(B)` | kỳ vọng `n/(n-B)` |
| 2. Ánh xạ lại (dict) | `O(B)` | `O(1)` | `O(B)` | 1 |
| 3. Nhị phân trên danh sách đen | `O(B log B)` | `O(log B)` | `O(B)` | 1 |

### 7.1. Cách 2 — Ánh xạ lại (remapping)

In [11]:
class PickRemap:
    def __init__(self, n, blacklist):
        self.m = n - len(blacklist)
        black = set(blacklist)
        self.remap = {}
        w = self.m                       # con trỏ tìm số trắng trong [m, n)
        for b in blacklist:
            if b < self.m:               # chỉ các lỗ trong [0, m) cần đổi tên
                while w in black:        # bỏ qua số đen trong [m, n)
                    w += 1
                self.remap[b] = w
                w += 1

    def f(self, k):                      # song ánh [0, m) -> số trắng (tách riêng để test)
        return self.remap.get(k, k)

    def pick(self):
        return self.f(random.randrange(self.m))

p = PickRemap(7, [2, 3, 5])
print("m =", p.m, "| remap =", p.remap, "| f:", [p.f(k) for k in range(p.m)])
random.seed(1)
print([p.pick() for _ in range(12)])

m = 4 | remap = {2: 4, 3: 6} | f: [0, 1, 4, 6]
[1, 0, 4, 0, 6, 6, 6, 6, 1, 0, 6, 0]


Giải thích từng dòng:

- `self.m = n - len(blacklist)`: số lượng số trắng = độ rộng vùng rút.
- `if b < self.m`: số đen ≥ `m` không bao giờ được rút ra (vì `k < m`) nên không cần làm gì.
- `while w in black: w += 1`: con trỏ `w` chỉ dừng ở số **trắng** trong `[m, n)`.
- `w += 1` sau khi gán: mỗi số trắng dư chỉ dùng **một lần** → đảm bảo song ánh.
- `pick`: đúng **một** lần gọi random.

Vì sao `w` không vượt quá `n - 1`? Số lỗ = số trắng dư (mục 4.2), nên đủ số dư để gán.

**Code nộp LeetCode (Cách 2):**

```python
import random

class Solution:
    def __init__(self, n: int, blacklist: List[int]):
        self.m = n - len(blacklist)
        black = set(blacklist)
        self.remap = {}
        w = self.m
        for b in blacklist:
            if b < self.m:
                while w in black:
                    w += 1
                self.remap[b] = w
                w += 1

    def pick(self) -> int:
        k = random.randrange(self.m)
        return self.remap.get(k, k)
```

**Độ phức tạp.** Khởi tạo `O(B)` (con trỏ `w` đi tối đa `B` bước tổng cộng trong `[m, n)` vì vùng đó chỉ có `B` ô); `pick` `O(1)`; bộ nhớ `O(B)`.

### 7.2. Cách 3 — Tìm số trắng thứ `k` bằng nhị phân

In [12]:
class PickBinarySearch:
    def __init__(self, n, blacklist):
        b = sorted(blacklist)
        self.m = n - len(b)
        self.key = [b[j] - j for j in range(len(b))]   # số trắng nhỏ hơn b[j]

    def f(self, k):                                    # số trắng thứ k (đếm từ 0)
        return k + bisect_right(self.key, k)

    def pick(self):
        return self.f(random.randrange(self.m))

p = PickBinarySearch(7, [2, 3, 5])
print("key =", p.key, "| f:", [p.f(k) for k in range(p.m)])
random.seed(1)
print([p.pick() for _ in range(12)])

key = [2, 2, 3] | f: [0, 1, 4, 6]
[1, 0, 4, 0, 6, 6, 6, 6, 1, 0, 6, 0]


**Vì sao `k + bisect_right(key, k)` đúng?** Gọi `c` = số số đen nhỏ hơn đáp án `x`. Khi đó `x = k + c` (trước `x` có `k` số trắng và `c` số đen). Số đen `b[j]` nằm trước `x` ⇔ số trắng nhỏ hơn `b[j]` (= `key[j]`) ≤ `k` (vì nếu `key[j] ≤ k` thì `b[j]` đứng trước số trắng thứ `k`). Vì `key` không giảm, các `j` thoả mãn là một tiền tố, độ dài `bisect_right(key, k)`.

**Code nộp LeetCode (Cách 3):**

```python
import random
from bisect import bisect_right

class Solution:
    def __init__(self, n: int, blacklist: List[int]):
        b = sorted(blacklist)
        self.m = n - len(b)
        self.key = [b[j] - j for j in range(len(b))]

    def pick(self) -> int:
        k = random.randrange(self.m)
        return k + bisect_right(self.key, k)
```

**Độ phức tạp.** Khởi tạo `O(B log B)` (sắp xếp); `pick` `O(log B)`; bộ nhớ `O(B)`. Ưu điểm: không cần bảng băm, và hàm `f` **đơn điệu tăng** (số trắng thứ `k` theo thứ tự) — tiện cho các biến thể như "rút số trắng thứ k trong một đoạn".

## 8. TRACE từng bước

### 8.1. Bảng trạng thái khi xây ánh xạ (Cách 2) với `n = 10`, `blacklist = [9, 1, 7, 3, 8]`

`m = 10 - 5 = 5`. Số đen trong `[0, 5)`: 1, 3. Vùng `[5, 10)`: 5 W, 6 W, 7 B, 8 B, 9 B.

| Số đen `b` (theo thứ tự đề) | `b < m`? | `w` trước | bỏ qua đen | gán | `w` sau |
|---|---|---|---|---|---|
| 9 | không | 5 | — | — | 5 |
| 1 | có | 5 | (5 trắng) | `1 → 5` | 6 |
| 7 | không | 6 | — | — | 6 |
| 3 | có | 6 | (6 trắng) | `3 → 6` | 7 |
| 8 | không | 7 | — | — | 7 |

`f = [0, 5, 2, 6, 4]` → tập ảnh `{0, 2, 4, 5, 6}` = đúng các số trắng.

### 8.2. `trace_remap`

In [13]:
def trace_remap(n, blacklist):
    m = n - len(blacklist)
    black = set(blacklist)
    print(f"n={n}, blacklist={blacklist}, m={m}")
    remap, w = {}, m
    for b in blacklist:
        if b >= m:
            print(f"  b={b}: >= m, bỏ qua")
            continue
        skipped = []
        while w in black:
            skipped.append(w)
            w += 1
        remap[b] = w
        print(f"  b={b}: lỗ trong [0,m); bỏ qua số đen {skipped}; gán {b} -> {w}")
        w += 1
    f = [remap.get(k, k) for k in range(m)]
    print("  f(k) cho k=0..m-1:", f)
    white = sorted(set(range(n)) - black)
    print("  tập ảnh == tập trắng?", sorted(f) == white)
    return remap

trace_remap(10, [9, 1, 7, 3, 8])
print()
trace_remap(7, [2, 3, 5])

n=10, blacklist=[9, 1, 7, 3, 8], m=5
  b=9: >= m, bỏ qua
  b=1: lỗ trong [0,m); bỏ qua số đen []; gán 1 -> 5
  b=7: >= m, bỏ qua
  b=3: lỗ trong [0,m); bỏ qua số đen []; gán 3 -> 6
  b=8: >= m, bỏ qua
  f(k) cho k=0..m-1: [0, 5, 2, 6, 4]
  tập ảnh == tập trắng? True

n=7, blacklist=[2, 3, 5], m=4
  b=2: lỗ trong [0,m); bỏ qua số đen []; gán 2 -> 4
  b=3: lỗ trong [0,m); bỏ qua số đen [5]; gán 3 -> 6
  b=5: >= m, bỏ qua
  f(k) cho k=0..m-1: [0, 1, 4, 6]
  tập ảnh == tập trắng? True


{2: 4, 3: 6}

### 8.3. `trace_bsearch`

In [14]:
def trace_bsearch(n, blacklist):
    b = sorted(blacklist)
    m = n - len(b)
    key = [b[j] - j for j in range(len(b))]
    print(f"n={n}, đen (sắp xếp)={b}, m={m}, key=b[j]-j={key}")
    for k in range(m):
        c = bisect_right(key, k)
        print(f"  k={k}: số key <= {k} là c={c} -> số trắng thứ {k} = {k}+{c} = {k + c}")

trace_bsearch(10, [9, 1, 7, 3, 8])

n=10, đen (sắp xếp)=[1, 3, 7, 8, 9], m=5, key=b[j]-j=[1, 2, 5, 5, 5]
  k=0: số key <= 0 là c=0 -> số trắng thứ 0 = 0+0 = 0
  k=1: số key <= 1 là c=1 -> số trắng thứ 1 = 1+1 = 2
  k=2: số key <= 2 là c=2 -> số trắng thứ 2 = 2+2 = 4
  k=3: số key <= 3 là c=2 -> số trắng thứ 3 = 3+2 = 5
  k=4: số key <= 4 là c=2 -> số trắng thứ 4 = 4+2 = 6


### 8.4. Trace một chuỗi `pick` với seed cố định

In [15]:
def trace_picks(cls, n, blacklist, times=8, seed=710):
    random.seed(seed)
    obj = cls(n, blacklist)
    out = []
    for _ in range(times):
        state = random.getstate()
        k = random.randrange(obj.m)
        random.setstate(state)               # quay lại để pick() rút đúng số k đó
        x = obj.pick()
        out.append((k, x))
    print(cls.__name__, "(k rút được, giá trị trả về):", out)

trace_picks(PickRemap, 10, [9, 1, 7, 3, 8])
trace_picks(PickBinarySearch, 10, [9, 1, 7, 3, 8])

PickRemap (k rút được, giá trị trả về): [(3, 6), (4, 4), (3, 6), (1, 5), (3, 6), (2, 2), (4, 4), (0, 0)]
PickBinarySearch (k rút được, giá trị trả về): [(3, 5), (4, 6), (3, 5), (1, 2), (3, 5), (2, 4), (4, 6), (0, 0)]


Hai cách cho cùng `k` nhưng có thể trả **giá trị khác nhau** (song ánh khác nhau) — cả hai đều đúng vì đều là song ánh sang tập trắng.

## 9. Code nộp LeetCode

Chọn **Cách 2** (ánh xạ lại): `pick` `O(1)`, đúng một lần gọi random.

In [16]:
from typing import List
import random

class Solution:

    def __init__(self, n: int, blacklist: List[int]):
        self.m = n - len(blacklist)
        black = set(blacklist)
        self.remap = {}
        w = self.m
        for b in blacklist:
            if b < self.m:
                while w in black:
                    w += 1
                self.remap[b] = w
                w += 1

    def pick(self) -> int:
        k = random.randrange(self.m)
        return self.remap.get(k, k)

random.seed(7)
obj = Solution(7, [2, 3, 5])
outs = [obj.pick() for _ in range(7)]
print("7 lần pick:", outs)
assert all(x in {0, 1, 4, 6} for x in outs)
print("Ví dụ đề: mọi kết quả hợp lệ")

7 lần pick: [4, 1, 6, 0, 0, 0, 4]
Ví dụ đề: mọi kết quả hợp lệ


## 10. Vì sao thuật toán đúng

**Song ánh (Cách 2).** Định nghĩa `f(k) = remap.get(k, k)` trên `[0, m)`.

- Nếu `k` trắng (`k < m`, `k ∉ black`): `f(k) = k`, là số trắng trong `[0, m)`.
- Nếu `k` đen (`k < m`): `f(k) = w_k` — một số trắng trong `[m, n)`; các `w_k` khác nhau vì con trỏ `w` chỉ tăng và tăng sau mỗi lần gán.

Ảnh của `f` = (số trắng trong `[0, m)`) ∪ (một số số trắng trong `[m, n)`), hai phần rời nhau, `f` đơn ánh. Kích thước miền = `m` = tổng số số trắng ⇒ `f` là song ánh lên **toàn bộ** tập trắng. (Số lỗ = số trắng trong `[m, n)` nên các số trắng dư được dùng hết.)

**Phân phối đều.** `k = randrange(m)` đều trên `[0, m)`; `f` song ánh ⇒ `pick()` đều trên tập trắng (mục 3.2).

**Cách 3.** `f(k)` = số trắng thứ `k` theo thứ tự tăng — rõ ràng là song ánh từ `[0, m)` lên tập trắng; chứng minh công thức ở mục 7.2.

**Một lần gọi random** mỗi `pick` ⇒ thoả yêu cầu tối ưu.

## 11. Phân tích Time/Space

| Cách | Khởi tạo | `pick` | Bộ nhớ |
|---|---|---|---|
| 1. Danh sách trắng | `O(n)` | `O(1)` | `O(n)` |
| 1b. Rút-loại | `O(B)` | kỳ vọng `O(n / (n - B))` | `O(B)` |
| 2. Ánh xạ lại | `O(B)` | `O(1)` | `O(B)` |
| 3. Nhị phân | `O(B log B)` | `O(log B)` | `O(B)` |

Đo thử với `n = 10^9`, `B = 10^5` (Cách 1 không thể chạy với `n` này):

In [17]:
import time

random.seed(11)
n_big = 10**9
bl_big = random.sample(range(n_big), 10**5)
for cls in [PickRemap, PickBinarySearch]:
    t0 = time.perf_counter()
    obj = cls(n_big, bl_big)
    t1 = time.perf_counter()
    for _ in range(20000):
        obj.pick()
    t2 = time.perf_counter()
    print(f"{cls.__name__:<17}: khởi tạo {t1 - t0:.3f}s, 20000 pick {t2 - t1:.3f}s")

# Rút-loại khi B gần n: đếm số lần gọi random
class CountingRejection(PickRejection):
    calls = 0
    def pick(self):
        while True:
            CountingRejection.calls += 1
            x = random.randrange(self.n)
            if x not in self.black:
                return x

random.seed(12)
n_small = 2000
obj = CountingRejection(n_small, list(range(1, n_small)))   # chỉ số 0 là trắng
for _ in range(50):
    obj.pick()
print("rút-loại, n=2000, B=1999: trung bình", CountingRejection.calls / 50, "lần gọi random mỗi pick")

PickRemap        : khởi tạo 0.019s, 20000 pick 0.005s
PickBinarySearch : khởi tạo 0.020s, 20000 pick 0.010s
rút-loại, n=2000, B=1999: trung bình 1820.2 lần gọi random mỗi pick


## 12. Bộ test

Chương trình ngẫu nhiên được kiểm thử bằng ba lớp:

1. **Kiểm tra tất định — song ánh:** với mọi `(n, blacklist)` nhỏ ngẫu nhiên (seed cố định), tập `{f(k) : k ∈ [0, m)}` phải **bằng đúng** tập trắng và không lặp. Đây là kiểm tra mạnh nhất vì không phụ thuộc may rủi.
2. **Mọi kết quả `pick` hợp lệ:** seed cố định, rút nhiều lần, mọi giá trị phải trắng và trong `[0, n)`.
3. **Phân phối thô:** rút `N` lần, tần suất mỗi số trắng phải gần `N / m` (cho phép lệch ±10%).

In [18]:
def white_set(n, blacklist):
    return set(range(n)) - set(blacklist)

# 1. Song ánh (tất định) — so với danh sách trắng (brute force)
random.seed(2029)
for _ in range(2000):
    n = random.randint(1, 30)
    B = random.randint(0, n - 1)
    bl = random.sample(range(n), B)
    W = white_set(n, bl)
    for cls in [PickRemap, PickBinarySearch]:
        obj = cls(n, bl)
        imgs = [obj.f(k) for k in range(obj.m)]
        assert len(imgs) == len(W) and set(imgs) == W, (cls.__name__, n, bl, imgs)
    sol = Solution(n, bl)
    assert {sol.remap.get(k, k) for k in range(sol.m)} == W
    assert sorted(PickWhitelist(n, bl).white) == sorted(W)
print("2000 bộ: ánh xạ là song ánh lên tập trắng (Cách 2, 3, Solution)")

# Edge case
assert PickRemap(1, []).f(0) == 0
assert PickRemap(5, [0, 1, 2, 3]).f(0) == 4           # chỉ còn một số trắng
assert PickBinarySearch(5, [0, 1, 2, 3]).f(0) == 4
assert [PickRemap(6, [4, 5]).f(k) for k in range(4)] == [0, 1, 2, 3]   # mọi số đen đều >= m
print("Edge case: OK")

2000 bộ: ánh xạ là song ánh lên tập trắng (Cách 2, 3, Solution)
Edge case: OK


In [19]:
# 2 + 3. pick hợp lệ và phân phối thô (seed cố định)
def check_distribution(cls, n, bl, N=40000, tol=0.10, seed=0):
    random.seed(seed)
    obj = cls(n, bl)
    W = white_set(n, bl)
    cnt = {}
    for _ in range(N):
        x = obj.pick()
        assert x in W, (cls.__name__, x)
        cnt[x] = cnt.get(x, 0) + 1
    expected = N / len(W)
    worst = max(abs(cnt.get(w, 0) - expected) / expected for w in W)
    assert set(cnt) == W and worst < tol, (cls.__name__, worst)
    return worst

for cls in [PickWhitelist, PickRejection, PickRemap, PickBinarySearch, Solution]:
    w1 = check_distribution(cls, 7, [2, 3, 5], seed=1)
    w2 = check_distribution(cls, 20, [0, 3, 4, 7, 11, 12, 13, 19], seed=2)
    print(f"{cls.__name__:<17}: lệch tương đối lớn nhất {w1:.3f}, {w2:.3f} (< 0.10)")

PickWhitelist    : lệch tương đối lớn nhất 0.015, 0.032 (< 0.10)


PickRejection    : lệch tương đối lớn nhất 0.010, 0.032 (< 0.10)


PickRemap        : lệch tương đối lớn nhất 0.015, 0.032 (< 0.10)


PickBinarySearch : lệch tương đối lớn nhất 0.015, 0.032 (< 0.10)
Solution         : lệch tương đối lớn nhất 0.015, 0.032 (< 0.10)


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `randint(0, m)` (gồm cả `m`)

In [20]:
class PickRandintBug(PickRemap):
    def pick(self):
        return self.f(random.randint(0, self.m))       # SAI: có thể ra m

random.seed(3)
obj = PickRandintBug(7, [2, 3, 5])
outs = {obj.pick() for _ in range(2000)}
print("tập kết quả:", sorted(outs), "| có số đen?", bool(outs & {2, 3, 5}))

tập kết quả: [0, 1, 4, 6] | có số đen? False


**Vì sao sai:** `randint(0, m)` có thể trả `m = 4`, không có trong `remap` → trả về 4 (tình cờ trắng, nhưng 4 bị trả về **gấp đôi** tần suất vì cũng là ảnh của `k = 2`). Với dữ liệu khác, `m` có thể là số đen. Dùng `randrange(m)` hoặc `randint(0, m - 1)`.

In [21]:
def bug_frequency():
    random.seed(4)
    obj = PickRandintBug(7, [2, 3, 5])
    c = {}
    for _ in range(50000):
        x = obj.pick()
        c[x] = c.get(x, 0) + 1
    return {k: round(v / 50000, 3) for k, v in sorted(c.items())}

print("tần suất (mong muốn 0.25 mỗi số):", bug_frequency())
bug = PickRandintBug(5, [3, 4])
print("n=5, blacklist=[3,4]: m =", bug.m, "| randint(0, m) có thể ra 3 -> f(3) =", bug.f(3), "| là số đen?", bug.f(3) in {3, 4})

tần suất (mong muốn 0.25 mỗi số): {0: 0.199, 1: 0.202, 4: 0.399, 6: 0.2}
n=5, blacklist=[3,4]: m = 3 | randint(0, m) có thể ra 3 -> f(3) = 3 | là số đen? True


### Lỗi 2: Ánh xạ lỗ sang `w` mà quên bỏ qua số đen trong `[m, n)`

In [22]:
class PickNoSkip:
    def __init__(self, n, blacklist):
        self.m = n - len(blacklist)
        self.remap = {}
        w = self.m
        for b in blacklist:
            if b < self.m:
                self.remap[b] = w                       # SAI: w có thể là số đen
                w += 1
    def f(self, k):
        return self.remap.get(k, k)

obj = PickNoSkip(7, [2, 5, 3])
print("f:", [obj.f(k) for k in range(obj.m)], "| chứa số đen 5?", 5 in [obj.f(k) for k in range(obj.m)])

f: [0, 1, 4, 5] | chứa số đen 5? True


**Vì sao sai:** vùng `[m, n)` cũng có số đen (ở đây 5). Phải `while w in black: w += 1` trước khi gán. Code đúng: `PickRemap`.

### Lỗi 3: Rút trong `[0, n)` rồi "đẩy" số đen sang số trắng kế tiếp

In [23]:
class PickShiftBug:
    def __init__(self, n, blacklist):
        self.n, self.black = n, set(blacklist)
    def pick(self):
        x = random.randrange(self.n)
        while x in self.black:                          # SAI: đẩy sang phải (vòng tròn)
            x = (x + 1) % self.n
        return x

random.seed(5)
obj = PickShiftBug(7, [2, 3, 5])
c = {}
for _ in range(70000):
    x = obj.pick()
    c[x] = c.get(x, 0) + 1
print("tần suất:", {k: round(v / 70000, 3) for k, v in sorted(c.items())}, "(mong muốn 0.25 mỗi số)")

tần suất: {0: 0.142, 1: 0.142, 4: 0.43, 6: 0.285} (mong muốn 0.25 mỗi số)


**Vì sao sai:** số 4 "hứng" luôn cả xác suất của 2 và 3 → `P(4) = 3/7`, không đều. Muốn đều thì phải là **song ánh** (mỗi số trắng đúng một nguồn), như Cách 2/3, hoặc rút lại hoàn toàn như rút-loại.

### Lỗi 4: Liệt kê danh sách trắng với `n = 10^9`

`[x for x in range(n) if …]` với một tỷ phần tử sẽ hết bộ nhớ/time limit. Luôn nhìn ràng buộc `n` trước khi chọn cách.

## 14. Mẹo & nhận dạng pattern

- **"Rút ngẫu nhiên đều trên một tập có lỗ"** → rút chỉ số `k ∈ [0, m)` rồi ánh xạ song ánh; tránh rút-loại khi tỉ lệ lỗ cao.
- **"Hoán đổi lỗ với phần dư ở cuối"** — cùng ý với mẹo xoá phần tử khỏi mảng trong `O(1)` bằng cách đổi với phần tử cuối (*Insert Delete GetRandom O(1)*, LeetCode 380).
- **"Phần tử thứ k không bị loại"** → `k + bisect_right(b[j] - j, k)` — mẹo hay gặp trong *Kth Missing Positive Number* (LeetCode 1539).
- Test chương trình ngẫu nhiên: **seed cố định**, tách phần **tất định** (hàm `f`) ra để kiểm tra song ánh, cộng thêm kiểm tra phân phối thô.

## 15. Biến thể

1. **LeetCode 528 — Random Pick with Weight:** xác suất tỉ lệ trọng số → prefix sum + `bisect`.
2. **LeetCode 398 — Random Pick Index:** chọn ngẫu nhiên một chỉ số có giá trị bằng `target` → reservoir sampling.
3. **LeetCode 497 — Random Point in Non-overlapping Rectangles:** prefix sum theo số điểm của từng hình chữ nhật.
4. **Danh sách đen thay đổi động** (thêm/bớt số đen) → cần cấu trúc động (Fenwick/Sorted list + nhị phân).
5. **Rút `c` số trắng khác nhau** → `random.sample(range(m), c)` rồi ánh xạ qua `f` (vẫn đều vì `f` song ánh).

Ví dụ biến thể 1 (Random Pick with Weight):

In [24]:
from itertools import accumulate

class PickWeight:
    def __init__(self, w):
        self.pre = list(accumulate(w))
    def pick_index(self):
        r = random.randrange(self.pre[-1])            # 0 .. tổng-1
        return bisect_right(self.pre, r)

random.seed(528)
pw = PickWeight([1, 3])
c = [0, 0]
for _ in range(40000):
    c[pw.pick_index()] += 1
print("tần suất:", [round(v / 40000, 3) for v in c], "(mong muốn 0.25, 0.75)")

tần suất: [0.251, 0.749] (mong muốn 0.25, 0.75)


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Danh sách trắng | Liệt kê số trắng, rút chỉ số | init `O(n)`, pick `O(1)` | `O(n)` | `n` nhỏ (≤ 10^6); làm trọng tài |
| 1b. Rút-loại | Rút `[0, n)`, đen thì rút lại | pick kỳ vọng `n/(n-B)` | `O(B)` | Danh sách đen rất thưa |
| 2. Ánh xạ lại | Lỗ trong `[0, m)` → số trắng dư trong `[m, n)` | init `O(B)`, pick `O(1)` | `O(B)` | Lời giải chuẩn, nhanh nhất |
| 3. Nhị phân | Số trắng thứ `k` = `k + bisect_right(key, k)` | init `O(B log B)`, pick `O(log B)` | `O(B)` | Khi cần thứ tự (f đơn điệu), không muốn dict |

## 17. Bài tập

**Bài 1 (đoán output).** Với `n = 6`, `blacklist = [0, 1]`: `m` = ? và dict `remap` của Cách 2 = ? Với `n = 4`, `blacklist = [3]`: `remap` = ?

**Bài 2 (tính tay).** Với `n = 12`, `blacklist = [10, 0, 4, 6, 11]`: tính `key` của Cách 3 và số trắng thứ `k` cho `k = 0..6`.

**Bài 3 (điền TODO).** Hoàn thiện `build_remap(n, blacklist)` trả về `(m, remap)`.

**Bài 4 (tự viết lại).** Viết lại lớp `MyPick` (Cách 3) với `__init__` và `pick`.

**Bài 5 (mini challenge).** Thêm phương thức `pick_many(c)` cho Cách 2: trả về `c` số trắng **khác nhau**, mọi tập con kích thước `c` đồng khả năng (gợi ý: `random.sample`).

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def build_remap(n, blacklist):
    # TODO: m = n - len(blacklist); w = m; với b < m: bỏ qua w đen, remap[b] = w, w += 1
    pass

# Bài 4 — TODO
class MyPick:
    def __init__(self, n, blacklist):
        # TODO: sắp blacklist; self.m; self.key = [b[j] - j ...]
        pass
    def pick(self):
        # TODO: k = randrange(m); return k + bisect_right(key, k)
        pass

# Bài 5 — TODO
def pick_many(obj, c):
    # TODO: [obj.f(k) for k in random.sample(range(obj.m), c)]
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `m = n - B`.
- Mức 2: `n=6, bl=[0,1]`: lỗ 0, 1; số trắng dư trong `[4, 6)`: 4, 5.
- Mức 3: `m = 4`, `remap = {0: 4, 1: 5}`; `n=4, bl=[3]`: `m = 3`, 3 ≥ m → `remap = {}`.

**Bài 2.**
- Mức 1: sắp xếp: `[0, 4, 6, 10, 11]`.
- Mức 2: `key = [0, 3, 4, 7, 7]`.
- Mức 3: số trắng: `k=0 → 0 + 1 = 1`, `1 → 2`, `2 → 3`, `3 → 3 + 2 = 5`, `4 → 4 + 3 = 7`, `5 → 8`, `6 → 9`.

**Bài 3.**
- Mức 1: `black = set(blacklist)`.
- Mức 2: duyệt các `b < m`.
- Mức 3: `while w in black: w += 1` trước khi gán.

**Bài 4.**
- Mức 1: `b = sorted(blacklist)`.
- Mức 2: `self.key = [b[j] - j for j in range(len(b))]`.
- Mức 3: `return k + bisect_right(self.key, k)`.

**Bài 5.**
- Mức 1: `random.sample(range(m), c)` trả về `c` chỉ số khác nhau, đều.
- Mức 2: ánh xạ từng chỉ số qua `f`.
- Mức 3: `f` song ánh → các giá trị khác nhau và phân phối đều trên tập con.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
p = PickRemap(6, [0, 1])
assert p.m == 4 and p.remap == {0: 4, 1: 5}
p = PickRemap(4, [3])
assert p.m == 3 and p.remap == {}

# Bài 2
p = PickBinarySearch(12, [10, 0, 4, 6, 11])
assert p.key == [0, 3, 4, 7, 7]
assert [p.f(k) for k in range(7)] == [1, 2, 3, 5, 7, 8, 9]

# Bài 3
def build_remap_ans(n, blacklist):
    m = n - len(blacklist)
    black = set(blacklist)
    remap, w = {}, m
    for b in blacklist:
        if b < m:
            while w in black:
                w += 1
            remap[b] = w
            w += 1
    return m, remap

assert build_remap_ans(7, [2, 3, 5]) == (4, {2: 4, 3: 6})
print("Bài 1–3 OK")

Bài 1–3 OK


In [27]:
# Bài 4
class MyPickAns:
    def __init__(self, n, blacklist):
        b = sorted(blacklist)
        self.m = n - len(b)
        self.key = [b[j] - j for j in range(len(b))]
    def pick(self):
        k = random.randrange(self.m)
        return k + bisect_right(self.key, k)

check_distribution(MyPickAns, 20, [0, 3, 4, 7, 11, 12, 13, 19], seed=9)

# Bài 5
def pick_many_ans(obj, c):
    return [obj.f(k) for k in random.sample(range(obj.m), c)]

random.seed(10)
obj = PickRemap(10, [9, 1, 7, 3, 8])
W = white_set(10, [9, 1, 7, 3, 8])
from collections import Counter
cnt = Counter()
for _ in range(20000):
    s = pick_many_ans(obj, 2)
    assert len(set(s)) == 2 and set(s) <= W
    cnt[frozenset(s)] += 1
expected = 20000 / 10                                  # C(5, 2) = 10 tập con
assert len(cnt) == 10 and max(abs(v - expected) / expected for v in cnt.values()) < 0.1
print("Bài 4–5 OK (phân phối thô đạt)")

Bài 4–5 OK (phân phối thô đạt)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template ánh xạ lại:**

```python
m = n - len(black); bset = set(black); remap = {}; w = m
for b in black:
    if b < m:
        while w in bset: w += 1
        remap[b] = w; w += 1
pick: k = randrange(m); return remap.get(k, k)
```

**Template số trắng thứ k:**

```python
b = sorted(black); key = [b[j] - j for j in range(len(b))]
pick: k = randrange(m); return k + bisect_right(key, k)
```

**Checklist trước khi submit:**

- [ ] Rút trong `[0, m)` với `m = n - len(blacklist)` — dùng `randrange(m)`, **không** `randint(0, m)`.
- [ ] Chỉ ánh xạ các số đen `< m`.
- [ ] Bỏ qua số đen khi chọn số trắng dư trong `[m, n)`.
- [ ] Mỗi số trắng dư dùng đúng một lần (song ánh).
- [ ] Không liệt kê `range(n)` khi `n` tới `10^9`.
- [ ] Test bằng seed cố định + kiểm tra song ánh + phân phối thô.
- [ ] Tên lớp/method đúng: `Solution(n, blacklist)`, `pick()`.